### CUDA 버전 확인하기

- !nvidia-smi
- !nvcc --version

둘의 차이점은
nvidia-smi는 해당 장치에서 설치 가능한 가장 높은 버전을 보여주고,
nvcc --version은 현재 설치된 cuda 버전을 보여줌

출처 : https://stackoverflow.com/questions/9727688/how-to-get-the-cuda-version

In [ ]:
gpu_info = !nvcc --version

In [ ]:
gpu_info = '\n'.join(gpu_info)
print(gpu_info)

In [ ]:
!nvidia-smi

In [1]:
# PyTorch 2.x 버전 설치
try:
    # 기본적으로 https://pytorch.kr/get-started/locally/ 에서
    # cuda 버전과 패키지매니저에 맞는 설치 명령어를 확인 가능
    # %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    import torch
    print(torch.__version__)
except:
    pass

2.11.0+cpu


----------------------------------------------------------

# VGGNet 직접 구현하기

In [2]:
# pytorch에서 사용할 함수들 호출하기
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

In [ ]:
# 데이터가 상당히 크기 때문에 시간이 좀 소요됨
transform = transforms.Compose([
    transforms.Resize(72),
    transforms.RandomCrop(56),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5 ), (0.5, 0.5, 0.5 )),
])

train_dataset = datasets.STL10(root='./data', split='train', download=True, transform=transform)
# train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)

test_dataset = datasets.STL10(root='./data', split='test', download=True, transform=transform)
# test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=False)

In [3]:
class VGGNet11(nn.Module):

    def __init__(self, num_classes=10):
        super(VGGNet11, self).__init__()
        # 3 x 56 x 56
        self.convnet = nn.Sequential(
            # 여기에 CNN 모델을 구현해주세요.
            nn.Conv2d(3, 64, 3, stride=1, padding=1), # 64 x 56 x 56
            nn.ReLU(),
            nn.MaxPool2d(2), # 64 x 28 x 28

            nn.Conv2d(64, 128, 3, stride=1, padding=1), # 128 x 28 x 28
            nn.ReLU(),
            nn.MaxPool2d(2), # 128 x 14 x 14

            nn.Conv2d(128, 256, 3, stride=1, padding=1), # 256 x 14 x 14
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), # 256 x 14 x 14
            nn.ReLU(),
            nn.MaxPool2d(2), # 256 x 7 x 7

            nn.Conv2d(256, 512, 3, stride=1, padding=1), # 512 x 7 x 7
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, stride=1, padding=1), # 512 x 7 x 7
            nn.ReLU(),
            nn.MaxPool2d(2), # 512 x 3 x 3

            nn.Conv2d(512, 512, 3, stride=1, padding=1), # 512 x 3 x 3
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, stride=1, padding=1), # 512 x 3 x 3
            nn.ReLU(),
            nn.MaxPool2d(2), # 512 x 1 x 1
        )

        self.fclayer = nn.Sequential(
            # 여기에 FC 모델을 구현해주세요.
            nn.Linear(512, 4096),
            nn.ReLU(),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.convnet(x)
        x = torch.flatten(x, 1)
        output = self.fclayer(x)
        return output

In [4]:
model = VGGNet11(num_classes=10)

In [5]:
%pip install torchinfo
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                   Param #
VGGNet11                                 --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       1,792
│    └─ReLU: 2-2                         --
│    └─MaxPool2d: 2-3                    --
│    └─Conv2d: 2-4                       73,856
│    └─ReLU: 2-5                         --
│    └─MaxPool2d: 2-6                    --
│    └─Conv2d: 2-7                       295,168
│    └─ReLU: 2-8                         --
│    └─Conv2d: 2-9                       590,080
│    └─ReLU: 2-10                        --
│    └─MaxPool2d: 2-11                   --
│    └─Conv2d: 2-12                      1,180,160
│    └─ReLU: 2-13                        --
│    └─Conv2d: 2-14                      2,359,808
│    └─ReLU: 2-15                        --
│    └─MaxPool2d: 2-16                   --
│    └─Conv2d: 2-17                      2,359,808
│    └─ReLU: 2-18                        --
│    └─Conv2d: 2-19              

In [ ]:
# 학습을 위한 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

criterion = nn.CrossEntropyLoss().cuda()
optimizer = optim.Adam(model.parameters(),lr=1e-5)
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

In [ ]:
# 모델 학습
epochs = 40
dry_run = False # 1 배치만 훈련

for epoch in range(1, epochs+1):
    # 학습
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 10 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.detach().cpu().item()))
            if dry_run:
                break

    # 테스트
    model.eval()
    test_loss = 0
    correct = 0

    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        with torch.no_grad():
            output = model(data)
        test_loss += criterion(output, target).detach().cpu().item()
        pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
        correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

    scheduler.step()


In [ ]:
for name, param in model.named_parameters():
    print(name)
    print(param)
    break